**Purpose:** Assemble the news sentiment training dataset from the labeling batches.

**Outputs:** `dataset.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


In [1]:
import pandas as pd
import os
import numpy as np

labeled_news_dir = str(PROJECT_ROOT / "02_sentiment/labeling")
news_quotes_path = str(PROJECT_ROOT / "01_data/processed/cache/news00001.parquet")

In [2]:
labels = [x for x in os.listdir(labeled_news_dir) if x.endswith(".parquet") and x.startswith("news_")]
print(f"True : Found {len(labels)} labeled files.")

expected_len = sum([len(df) for df in [pd.read_parquet(os.path.join(labeled_news_dir, df_name)) for df_name in labels]])
concated_df = pd.concat([x for x in [pd.read_parquet(os.path.join(labeled_news_dir, df_name)) for df_name in labels]], ignore_index=True)
print(f"{expected_len==len(concated_df)} : Validate expected length of concatenated DataFrame")

GKGRECORDID_above_3 = sum(pd.Series(concated_df["GKGRECORDID"].value_counts().values) > 3)
print(f"{GKGRECORDID_above_3 == 0} : Validate no ({GKGRECORDID_above_3}) GKGRECORDID appears more than 3 times")

df_grouped = (
    concated_df.groupby("GKGRECORDID", as_index=False)
      .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
)
original_len = len(df_grouped)
df_grouped.dropna(inplace=True)
dropped_len = original_len - len(df_grouped)
print(f"True : Dropped {dropped_len} rows with NaN values after grouping.")


df_grouped

True : Found 42 labeled files.
True : Validate expected length of concatenated DataFrame
False : Validate no (3) GKGRECORDID appears more than 3 times
True : Dropped 12254 rows with NaN values after grouping.


,GKGRECORDID,google/gemma-7b-it,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-7B-Instruct
18,20150705141500-1029 (Consumer Discretionary),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 0.8, 'rationale': 'The quote me...","{'confidence': 0.8, 'rationale': 'The quote di..."
19,20150705141500-119 (Information Technology),"{'confidence': 0.1, 'error': None, 'news_id': ...","{'confidence': 1.0, 'rationale': 'The event de...","{'confidence': 0.8, 'rationale': 'The quote de..."
20,20150705141500-1254 (Consumer Discretionary),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 0.8, 'rationale': 'The news quo...","{'confidence': 0.8, 'rationale': 'The quote di..."
21,20150705141500-1254 (Energy),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 0.9, 'rationale': 'The quote di...","{'confidence': 0.8, 'rationale': 'The quote di..."
22,20150705141500-1254 (Financials),"{'confidence': 0.0, 'error': None, 'news_id': ...","{'confidence': 0.8, 'rationale': 'The agreemen...","{'confidence': 0.8, 'rationale': 'The quote di..."
...,...,...,...,...
58978,20250614003000-616 (Real Estate),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 0.9, 'rationale': 'The quote me...","{'confidence': 1.0, 'rationale': 'The quote 'b..."
58979,20250614003000-635 (Financials),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 1.0, 'rationale': 'The quote do...","{'confidence': 1.0, 'rationale': 'The quote do..."
58980,20250614003000-635 (Health Care),"{'confidence': 0.8, 'error': None, 'news_id': ...","{'confidence': 0.8, 'rationale': 'The event su...","{'confidence': 0.8, 'rationale': 'The quote su..."
58981,20250614003000-780 (Health Care),"{'confidence': 0.2, 'error': None, 'news_id': ...","{'confidence': 0.9, 'rationale': 'This news qu...","{'confidence': 1.0, 'rationale': 'The quote di..."


In [3]:
cached_quotes = pd.read_parquet(news_quotes_path)

cached_quotes

,id,quotation
0,20150705141500-1029,Greek troops and police prepare for street bat...
1,20150705141500-119,the level 4 alert ( out of a maximum of 5 ) co...
2,20150705141500-1254,Must the agreement plan submitted by the Europ...
3,20150705141500-1296,"as an independent, united bloc"
4,20150705141500-155,because they ( the creditors ) will take us mo...
...,...,...
20969,20250611224500-429,near-term action recommendations no later than...
20970,20250611224500-501,unfortunately has determined to not continue S...
20971,20250611224500-588,Collegiate Solar Boating Competition
20972,20250611224500-66,advisory transitional services


In [4]:
sectors = {"Energy", "Materials", "Industrials", "Consumer Discretionary", "Consumer Staples", "Health Care", "Financials", "Information Technology", "Communication Services", "Utilities", "Real Estate"}

errors = {
    "sector_mismatch": [],
    "missing_quotation": [],
    "missing_model_output": [],
    "no_majority": [],
}

dataset = {
    "GKGRECORDID": [],
    "sector": [],
    "quotation": [],
    "majority_sentiment": [],
}

for i, row in df_grouped.iterrows():

    id, sector = row["GKGRECORDID"].split(maxsplit=1)
    sector = sector.replace("(", "").replace(")", "")
    quotation = cached_quotes.loc[cached_quotes["id"] == id, "quotation"].values

    if len(quotation) == 0:
        errors["missing_quotation"].append(id)
        continue
    quotation = quotation[0]
    
    a = row["google/gemma-7b-it"]
    b = row["Qwen/Qwen2.5-7B-Instruct"]
    c = row["meta-llama/Llama-3.1-8B-Instruct"]

    votes = {"positive": 0, "neutral": 0, "negative": 0}
    for model_output in [a, b, c]:
        if model_output is None:
            errors["missing_model_output"].append(id)
            continue
        
        if "sector" not in model_output or model_output["sector"] != sector:
            errors["sector_mismatch"].append(f"{id}:{model_output.get('sector', None)}, expected:{sector}")
            continue

        sentiment = model_output.get("sentiment", None)
        if sentiment is None:
            continue
        elif sentiment not in votes:
            errors["missing_model_output"].append(id)
            continue
        else:
            votes[sentiment] += 1


    if all(vote_count < 2 for vote_count in votes.values()):
        errors["no_majority"].append(f"{id}:{votes}")
        continue
    majority_sentiment = max(votes, key=votes.get)

    dataset["GKGRECORDID"].append(row["GKGRECORDID"])
    dataset["sector"].append(sector)
    dataset["quotation"].append(quotation)
    dataset["majority_sentiment"].append(majority_sentiment)

final_df = pd.DataFrame(dataset)
final_df.to_parquet("dataset.parquet", index=False)

In [5]:
final_df

,GKGRECORDID,sector,quotation,majority_sentiment
0,20150705141500-1029 (Consumer Discretionary),Consumer Discretionary,Greek troops and police prepare for street bat...,neutral
1,20150705141500-119 (Information Technology),Information Technology,the level 4 alert ( out of a maximum of 5 ) co...,neutral
2,20150705141500-1254 (Consumer Discretionary),Consumer Discretionary,Must the agreement plan submitted by the Europ...,neutral
3,20150705141500-1254 (Energy),Energy,Must the agreement plan submitted by the Europ...,neutral
4,20150705141500-1254 (Financials),Financials,Must the agreement plan submitted by the Europ...,neutral
...,...,...,...,...
46246,20250614003000-616 (Real Estate),Real Estate,"black service, not an NHS service",neutral
46247,20250614003000-635 (Financials),Financials,If the state knew that he had a mental health ...,neutral
46248,20250614003000-635 (Health Care),Health Care,If the state knew that he had a mental health ...,negative
46249,20250614003000-780 (Health Care),Health Care,We know that the turbines that will be the per...,neutral


In [6]:
final_df.value_counts(["sector", "majority_sentiment"])

sector                  majority_sentiment
Information Technology  neutral               7498
Health Care             neutral               7487
Real Estate             neutral               4043
Utilities               neutral               3117
Consumer Staples        neutral               3110
Energy                  neutral               2924
Communication Services  neutral               2656
Financials              neutral               2631
Industrials             neutral               2350
Consumer Discretionary  neutral               2136
Materials               neutral               1562
Financials              negative               525
Information Technology  positive               459
Health Care             negative               438
Industrials             positive               435
                        negative               412
Information Technology  negative               387
Health Care             positive               361
Real Estate             negative       